In [1]:
import pandas as pd
import numpy as np

df = pd.read_excel("/kaggle/input/datasets/karvysingh/dalisa/FCC_feed_data.xlsx",
                   sheet_name="Feed Data")

# remove the units row
df = df[pd.to_numeric(df["Sample No."], errors="coerce").notna()].copy()

df["Sample No."] = df["Sample No."].astype(int)

for c in df.columns[1:]:
    df[c] = pd.to_numeric(df[c])

df.head()

,Sample No.,Sp. Gr,CCR,T.S,T.N,IBP,5 %,10 %,30 %,50 %,70 %,90 %,95 %,FBP,P,N,A
1,1,0.9245,0.40,2.93,1401,37,317,342,392,423,454,498,514,538,8.3,24.1,67.6
2,2,0.9960,1.55,4.35,200,42,409,428,458,477,494,514,523,537,7.8,25.3,66.8
3,3,0.8683,0.04,0.03,22,103,224,263,337,380,420,473,497,536,15.2,45.0,39.8
4,4,0.9136,0.04,1.54,329,111,276,318,368,391,407,427,436,460,14.8,15.5,70.5
5,5,0.9303,0.28,3.06,1161,169,257,320,402,438,474,519,535,574,11.2,23.3,65.6


In [2]:
features = [
    "Sp. Gr",
    "5 %",
    "10 %",
    "30 %",
    "50 %",
    "70 %",
    "90 %",
    "95 %"
]

targets = ["P", "N", "A"]

# published validation samples
val = df[df["Sample No."].between(17, 22)].copy()

# remaining samples for development
dev = df[~df["Sample No."].between(17, 22)].copy()

X = dev[features]
y = dev[targets]

X_val = val[features]
y_val = val[targets]

In [3]:
train = df[df["Sample No."].between(1, 16)].copy()
test  = df[df["Sample No."].between(23, 28)].copy()
val   = df[df["Sample No."].between(17, 22)].copy()

X_train = train[features]
y_train = train[targets]

X_test = test[features]
y_test = test[targets]

X_val = val[features]
y_val = val[targets]

In [4]:
from sklearn.preprocessing import MinMaxScaler

x_scaler = MinMaxScaler(feature_range=(-1, 1))

X_train = x_scaler.fit_transform(train[features])
X_test  = x_scaler.transform(test[features])
X_val   = x_scaler.transform(val[features])

# separate target scaler for each ANN
p_scaler = MinMaxScaler(feature_range=(-1, 1))
n_scaler = MinMaxScaler(feature_range=(-1, 1))
a_scaler = MinMaxScaler(feature_range=(-1, 1))

P_train = p_scaler.fit_transform(
    train[["P"]]
).ravel()

N_train = n_scaler.fit_transform(
    train[["N"]]
).ravel()

A_train = a_scaler.fit_transform(
    train[["A"]]
).ravel()

In [5]:
import torch
import torch.nn as nn
import numpy as np

torch.manual_seed(42)

Xtr = torch.tensor(X_train, dtype=torch.float32)
Xte = torch.tensor(X_test, dtype=torch.float32)
Xv  = torch.tensor(X_val, dtype=torch.float32)

class ANN(nn.Module):
    def __init__(self, h1, h2):
        super().__init__()
        self.fc1 = nn.Linear(8, h1)
        self.fc2 = nn.Linear(h1, h2)
        self.out = nn.Linear(h2, 1)

    def forward(self, x):
        x = torch.tanh(self.fc1(x))      # tansig
        x = torch.sigmoid(self.fc2(x))  # logsig
        return self.out(x)               # purelin

In [6]:
def train_model(y, h1, h2):
    model = ANN(h1, h2)

    y = torch.tensor(
        np.asarray(y, dtype=np.float32).reshape(-1, 1)
    )

    optimizer = torch.optim.LBFGS(
        model.parameters(),
        lr=0.5,
        max_iter=1000,
        tolerance_grad=1e-9,
        tolerance_change=1e-12,
        line_search_fn="strong_wolfe"
    )

    loss_fn = nn.MSELoss()

    def closure():
        optimizer.zero_grad()
        pred = model(Xtr)
        loss = loss_fn(pred, y)
        loss.backward()
        return loss

    optimizer.step(closure)

    return model

In [7]:
model_P = train_model(y_train["P"], 12, 13)
model_N = train_model(y_train["N"], 12, 12)
model_A = train_model(y_train["A"], 9, 9)

with torch.no_grad():
    P_scaled = model_P(Xv).numpy().ravel()
    N_scaled = model_N(Xv).numpy().ravel()
    A_scaled = model_A(Xv).numpy().ravel()

P_raw = p_scaler.inverse_transform(
    P_scaled.reshape(-1, 1)
).ravel()

N_raw = n_scaler.inverse_transform(
    N_scaled.reshape(-1, 1)
).ravel()

A_raw = a_scaler.inverse_transform(
    A_scaled.reshape(-1, 1)
).ravel()

total = P_raw + N_raw + A_raw

P = 100 * P_raw / total
N = 100 * N_raw / total
A = 100 * A_raw / total

pred = np.column_stack([P, N, A])


In [8]:
result = pd.DataFrame({
    "Sample": val["Sample No."].values,
    "P_exp": y_val["P"].values,
    "P_pred": P,
    "N_exp": y_val["N"].values,
    "N_pred": N,
    "A_exp": y_val["A"].values,
    "A_pred": A,
})

display(result.round(2))

,Sample,P_exp,P_pred,N_exp,N_pred,A_exp,A_pred
0,17,15.8,7.52,35.9,32.759998,48.3,59.709999
1,18,14.4,7.15,23.3,21.260000,62.2,71.589996
2,19,4.9,7.18,21.1,18.670000,74.0,74.139999
3,20,10.9,16.59,29.9,22.620001,59.2,60.790001
4,21,10.4,7.33,18.5,18.340000,71.0,74.320000
5,22,6.3,6.14,26.2,21.020000,67.5,72.849998


In [9]:
from sklearn.metrics import mean_squared_error, r2_score

obs = y_val[["P", "N", "A"]].values.ravel()
prd = pred.ravel()

rmse = np.sqrt(mean_squared_error(obs, prd))
r2 = r2_score(obs, prd)

print("RMSE =", round(rmse, 3))
print("R²   =", round(r2, 4))

RMSE = 5.417
R²   = 0.9458
